# Hands-on 3: Mengirim Data Sensor ke MQTT

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* oleh Onno W. Purbo (Halaman 73–78).

## Tujuan

Mengirim data sensor ke broker MQTT menggunakan Python di Ubuntu 26.04. Hasil akhir:

- memasang dan menjalankan Mosquitto MQTT broker
- memahami konsep topic, publisher, subscriber, dan QoS
- membuat simulator sensor sederhana dengan Python
- mengirim data sensor dalam format JSON
- menguji bahwa pesan benar-benar masuk ke broker

Target konkret hands-on ini:
1. Memahami arsitektur *hub-and-spoke* MQTT untuk pengumpulan data sensor (telemetri).
2. Membangun publisher / simulator sensor IoT menggunakan pustaka `paho-mqtt` di Python.
3. Mengirimkan payload JSON terstruktur ke topik MQTT bertingkat secara periodik.
4. Menerapkan variasi realistis untuk kasus smart room dan greenhouse (pertanian presisi).


### 1. Konfigurasi Broker dan Pustaka Paho-MQTT
Dalam latihan ini, klien publisher akan terhubung ke broker Mosquitto di `localhost:1883`.


In [1]:
import json
import random
import time
from datetime import datetime
import paho.mqtt.client as mqtt

BROKER_HOST = "localhost"
BROKER_PORT = 1883
TOPIC_ROOM = "kampus/lab1/room01/telemetry"
CLIENT_ID = "notebook-room-publisher"

print(f"[CONFIG] Broker: {BROKER_HOST}:{BROKER_PORT}")
print(f"[CONFIG] Topik : {TOPIC_ROOM}")


[CONFIG] Broker: localhost:1883
[CONFIG] Topik : kampus/lab1/room01/telemetry


### 2. Generator Data Sensor Sintetis (Room Telemetry)
Fungsi membangkitkan data suhu, kelembaban, CO2, daya listrik, dan timestamp ISO-8601.


In [2]:
def generate_sensor_data(device_id="room01"):
    return {
        "device_id": device_id,
        "temperature_c": round(random.uniform(24.0, 30.0), 2),
        "humidity_pct": round(random.uniform(45.0, 65.0), 2),
        "co2_ppm": random.randint(450, 900),
        "power_watt": round(random.uniform(100.0, 160.0), 2),
        "timestamp": datetime.now().isoformat(timespec="seconds")
    }

contoh_payload = generate_sensor_data()
print("Contoh Payload JSON:")
print(json.dumps(contoh_payload, indent=2))


Contoh Payload JSON:
{
  "device_id": "room01",
  "temperature_c": 26.58,
  "humidity_pct": 50.54,
  "co2_ppm": 693,
  "power_watt": 126.16,
  "timestamp": "2026-11-02T12:07:03"
}


### 3. Eksekusi Pengiriman Pesan ke Broker MQTT
Membuka koneksi ke Mosquitto dan mempublikasikan 5 paket data secara periodik.


In [3]:
client = mqtt.Client(client_id=CLIENT_ID)

try:
    client.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    print(f"[INFO] Terhubung ke broker MQTT ({BROKER_HOST}:{BROKER_PORT})")

    for i in range(5):
        payload = generate_sensor_data()
        payload_json = json.dumps(payload)
        res = client.publish(TOPIC_ROOM, payload_json, qos=1)
        res.wait_for_publish(timeout=3)
        print(f"[{i+1}/5] Publish sukses ke {TOPIC_ROOM} -> {payload_json}")
        time.sleep(1)

    client.disconnect()
    print("[SELESAI] Pengiriman telemetri ruang selesai.")
except Exception as e:
    print(f"[ERROR] Koneksi MQTT gagal: {e}")


[INFO] Terhubung ke broker MQTT (localhost:1883)


[1/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 25.65, "humidity_pct": 51.63, "co2_ppm": 665, "power_watt": 158.08, "timestamp": "2026-11-02T12:07:03"}


[2/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 25.57, "humidity_pct": 48.11, "co2_ppm": 582, "power_watt": 102.65, "timestamp": "2026-11-02T12:07:07"}


[3/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 27.7, "humidity_pct": 56.15, "co2_ppm": 804, "power_watt": 149.79, "timestamp": "2026-11-02T12:07:11"}


[4/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 24.6, "humidity_pct": 46.56, "co2_ppm": 742, "power_watt": 140.61, "timestamp": "2026-11-02T12:07:15"}


[5/5] Publish sukses ke kampus/lab1/room01/telemetry -> {"device_id": "room01", "temperature_c": 28.81, "humidity_pct": 58.0, "co2_ppm": 890, "power_watt": 121.2, "timestamp": "2026-11-02T12:07:19"}


[SELESAI] Pengiriman telemetri ruang selesai.


### 4. Variasi Realistis: Sensor Pertanian Presisi (Greenhouse)
Mengirimkan parameter pertanian: suhu udara, kelembaban tanah (*soil moisture*), dan intensitas cahaya (*lux*).


In [4]:
TOPIC_GREENHOUSE = "kampus/greenhouse/zone1/telemetry"

def generate_greenhouse_data():
    return {
        "device_id": "greenhouse-zone1",
        "air_temp_c": round(random.uniform(25.0, 35.0), 2),
        "soil_moisture_pct": round(random.uniform(30.0, 80.0), 2),
        "light_lux": random.randint(8000, 30000),
        "timestamp": datetime.now().isoformat(timespec="seconds")
    }

client_gh = mqtt.Client(client_id="notebook-gh-pub")

try:
    client_gh.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    for i in range(3):
        gh_data = generate_greenhouse_data()
        gh_json = json.dumps(gh_data)
        client_gh.publish(TOPIC_GREENHOUSE, gh_json, qos=1)
        print(f"[GREENHOUSE {i+1}/3] {gh_json}")
        time.sleep(1)
    client_gh.disconnect()
    print("[SELESAI] Telemetri greenhouse terkirim.")
except Exception as e:
    print(f"[ERROR] Koneksi MQTT greenhouse gagal: {e}")


[GREENHOUSE 1/3] {"device_id": "greenhouse-zone1", "air_temp_c": 28.57, "soil_moisture_pct": 66.56, "light_lux": 23463, "timestamp": "2026-11-02T12:07:23"}


[GREENHOUSE 2/3] {"device_id": "greenhouse-zone1", "air_temp_c": 32.15, "soil_moisture_pct": 77.93, "light_lux": 27542, "timestamp": "2026-11-02T12:07:24"}


[GREENHOUSE 3/3] {"device_id": "greenhouse-zone1", "air_temp_c": 32.78, "soil_moisture_pct": 30.34, "light_lux": 18756, "timestamp": "2026-11-02T12:07:25"}


[SELESAI] Telemetri greenhouse terkirim.


## Solusi Tantangan (Tambahan di Luar Buku)

**Tantangan dari buku:**
- ganti localhost dengan IP Raspberry Pi atau server Ubuntu lain
- ubah topik menjadi beberapa ruangan: room01, room02, room03
- kirim data dengan interval berbeda
- tambahkan field battery_pct
- kirim status perangkat ke topik status
- buat satu publisher untuk suhu, satu lagi untuk energi

_Bagian ini BUKAN dari buku. Tantangan pada bagian sebelumnya dikerjakan di sini; kode buku tetap apa adanya._

Catatan untuk tantangan (1): mengganti `localhost` dengan IP Raspberry Pi atau server Ubuntu lain cukup dilakukan dengan mengubah nilai `BROKER_HOST` pada sel konfigurasi di atas, misalnya `BROKER_HOST = "192.168.1.10"`. Tidak dijalankan di sini karena membutuhkan perangkat lain di jaringan.


In [5]:
# [SOLUSI 2-5] Topik multi-ruangan, interval berbeda, battery_pct, dan topik status.
client_t = mqtt.Client(client_id="notebook-tantangan-pub")
rooms = ["room01", "room02", "room03"]          # (2) beberapa ruangan
interval_detik = {"room01": 0.2, "room02": 0.5, "room03": 1.0}  # (3) interval berbeda

try:
    client_t.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    print(f"[INFO] Terhubung ke broker ({BROKER_HOST}:{BROKER_PORT})")
    for dev in rooms:
        topic = f"kampus/lab1/{dev}/telemetry"
        for _ in range(2):
            payload = generate_sensor_data(device_id=dev)
            payload["battery_pct"] = random.randint(20, 100)  # (4) field battery_pct
            client_t.publish(topic, json.dumps(payload), qos=1)
            print(f"[{dev}] {json.dumps(payload)}")
            time.sleep(interval_detik[dev])
        status = {"device_id": dev, "status": "online",
                  "timestamp": datetime.now().isoformat(timespec="seconds")}
        client_t.publish(f"kampus/lab1/{dev}/status", json.dumps(status), qos=1)  # (5) topik status
        print(f"[{dev}] status -> {json.dumps(status)}")
    client_t.disconnect()
    print("[SELESAI] Telemetri 3 ruangan + status terkirim.")
except Exception as e:
    print(f"[ERROR] Koneksi MQTT gagal: {e}")

[INFO] Terhubung ke broker (localhost:1883)
[room01] {"device_id": "room01", "temperature_c": 24.25, "humidity_pct": 50.55, "co2_ppm": 606, "power_watt": 147.47, "timestamp": "2026-11-02T12:07:26", "battery_pct": 25}


[room01] {"device_id": "room01", "temperature_c": 29.9, "humidity_pct": 61.55, "co2_ppm": 884, "power_watt": 130.35, "timestamp": "2026-11-02T12:07:26", "battery_pct": 82}


[room01] status -> {"device_id": "room01", "status": "online", "timestamp": "2026-11-02T12:07:26"}
[room02] {"device_id": "room02", "temperature_c": 26.68, "humidity_pct": 50.76, "co2_ppm": 647, "power_watt": 120.71, "timestamp": "2026-11-02T12:07:26", "battery_pct": 68}


[room02] {"device_id": "room02", "temperature_c": 25.54, "humidity_pct": 45.33, "co2_ppm": 643, "power_watt": 126.7, "timestamp": "2026-11-02T12:07:27", "battery_pct": 94}


[room02] status -> {"device_id": "room02", "status": "online", "timestamp": "2026-11-02T12:07:27"}
[room03] {"device_id": "room03", "temperature_c": 28.1, "humidity_pct": 64.97, "co2_ppm": 889, "power_watt": 152.76, "timestamp": "2026-11-02T12:07:27", "battery_pct": 87}


[room03] {"device_id": "room03", "temperature_c": 27.88, "humidity_pct": 46.58, "co2_ppm": 601, "power_watt": 136.57, "timestamp": "2026-11-02T12:07:28", "battery_pct": 35}


[room03] status -> {"device_id": "room03", "status": "online", "timestamp": "2026-11-02T12:07:29"}
[SELESAI] Telemetri 3 ruangan + status terkirim.


In [6]:
# [SOLUSI 6] Dua publisher terpisah: satu untuk suhu, satu untuk energi.
pub_suhu = mqtt.Client(client_id="notebook-pub-suhu")
pub_energi = mqtt.Client(client_id="notebook-pub-energi")

try:
    pub_suhu.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    pub_energi.connect(BROKER_HOST, BROKER_PORT, keepalive=60)
    for i in range(3):
        suhu = {"device_id": "room01",
                "temperature_c": round(random.uniform(24.0, 30.0), 2),
                "timestamp": datetime.now().isoformat(timespec="seconds")}
        energi = {"device_id": "room01",
                  "power_watt": round(random.uniform(100.0, 160.0), 2),
                  "timestamp": datetime.now().isoformat(timespec="seconds")}
        pub_suhu.publish("kampus/lab1/room01/suhu", json.dumps(suhu), qos=1)
        pub_energi.publish("kampus/lab1/room01/energi", json.dumps(energi), qos=1)
        print(f"[SUHU] {json.dumps(suhu)}")
        print(f"[ENERGI] {json.dumps(energi)}")
        time.sleep(0.5)
    pub_suhu.disconnect()
    pub_energi.disconnect()
    print("[SELESAI] Publisher suhu dan energi berjalan terpisah.")
except Exception as e:
    print(f"[ERROR] Koneksi MQTT gagal: {e}")

[SUHU] {"device_id": "room01", "temperature_c": 25.23, "timestamp": "2026-11-02T12:07:29"}
[ENERGI] {"device_id": "room01", "power_watt": 159.38, "timestamp": "2026-11-02T12:07:29"}


[SUHU] {"device_id": "room01", "temperature_c": 28.8, "timestamp": "2026-11-02T12:07:30"}
[ENERGI] {"device_id": "room01", "power_watt": 129.31, "timestamp": "2026-11-02T12:07:30"}


[SUHU] {"device_id": "room01", "temperature_c": 27.26, "timestamp": "2026-11-02T12:07:30"}
[ENERGI] {"device_id": "room01", "power_watt": 119.58, "timestamp": "2026-11-02T12:07:30"}


[SELESAI] Publisher suhu dan energi berjalan terpisah.


## Penutup

Hasil running:
- [INFO] Terhubung ke broker (localhost:1883)
- [SELESAI] Telemetri 3 ruangan + status terkirim.
- [SELESAI] Publisher suhu dan energi berjalan terpisah.
